# FD002 and FD004 teacher tuning

Train the supplied Conv1D → bidirectional GRU → additive attention → regression architecture. Evaluate only teachers. The sweep varies observed history, hidden width, dropout, learning rate, causal sensor smoothing and training label cap. Select each checkpoint and configuration by raw validation RMSE on engines held out from training. Report raw and capped test RMSE separately after freezing the selections.

In [ ]:
from pathlib import Path
import hashlib
import subprocess
import sys
from urllib.request import urlretrieve
from zipfile import ZipFile
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime to run the sweep.")
drive.mount("/content/drive")
repo = Path("/content/ML3103-project-teacher-tuning")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/abhav-arora/ML3103-project.git", str(repo)], check=True)
data_root = Path("/content/cmapss_teacher_tuning")
data_root.mkdir(exist_ok=True)
subsets = ["FD002", "FD004"]
if not all((data_root / f"{kind}_{fd}.txt").exists() for fd in subsets for kind in ("train", "test", "RUL")):
    archive = data_root / "CMAPSSData.zip"
    urlretrieve("https://zenodo.org/records/15346912/files/CMAPSSData.zip?download=1", archive)
    assert hashlib.md5(archive.read_bytes()).hexdigest() == "79a22f36e80606c69d0e9e4da5bb2b7a"
    with ZipFile(archive) as source:
        names = {Path(name).name: name for name in source.namelist()}
        for fd in subsets:
            for kind in ("train", "test", "RUL"):
                name = f"{kind}_{fd}.txt"
                (data_root / name).write_bytes(source.read(names[name]))
output = Path("/content/drive/MyDrive/ML projec/SpikeDistill-RUL/colab_runs/additive_bigru_teacher_v1")
command = [sys.executable, "-u", "-m", "research_extensions.v2.tune_bigru",
           "--data-root", str(data_root), "--output", str(output)]

## Choose settings on validation

In [ ]:
subprocess.run([*command, "--phase", "tune"], cwd=repo, check=True)

## Evaluate selected teachers on the official test sets

In [ ]:
subprocess.run([*command, "--phase", "test"], cwd=repo, check=True)

import pandas as pd
scores = pd.read_csv(output / "teacher_test_comparison.csv").set_index("subset")
original = pd.read_csv(repo / "results/test_metrics_per_seed.csv")
original = original.query("target == 'raw' and seed == 42 and model != 'mean'")
original = original.loc[original.groupby("subset").RMSE.idxmin()].set_index("subset")
scores["best_original_model"] = original.model
scores["best_original_rmse"] = original.RMSE
v1 = pd.read_csv(repo / "research_extensions/results/teacher_student/colab/research_metrics.csv")
v1 = v1.query("split == 'test' and target == 'raw' and model in ['gru_memory', 'temporal_cnn', 'teacher_average']")
# Saved V2 raw scores: https://drive.google.com/file/d/1L8oQppfaxowJhtkzK_IUi6AKwSH3d00Q/view
v2_best = pd.Series({"FD002": 27.24749634095565, "FD004": 26.898122023979084})
scores["best_prior_teacher_rmse"] = pd.concat(
    [v1.groupby("subset").RMSE.min(), v2_best], axis=1
).min(axis=1)
scores["delta_vs_original"] = scores.test_raw_rmse - scores.best_original_rmse
scores["delta_vs_prior_teacher"] = scores.test_raw_rmse - scores.best_prior_teacher_rmse
display(scores.round(3))
scores.to_csv(output / "teacher_test_vs_prior.csv")